# Sparse Walker temporal-memory single-batch profiler

No dataset and no full epoch. This compares plain Walker and temporal Walker on the same synthetic B=128, L=200 ML-1M-shaped batch. Every stage synchronizes CUDA and prints immediately.


In [ ]:
import os, sys, subprocess, shutil, time, torch
REPO='/content/Sparsewalker'
BRANCH='agent/walker-temporal-skips'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
SRC=f'{REPO}/src'; sys.path.insert(0,SRC)
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'): del sys.modules[name]
from sparsewalker.models import SparseWalker, SparseWalkerTemporalMemory
from sparsewalker.models.core import ar_training_loss
assert torch.cuda.is_available()
device='cuda'; torch.manual_seed(42)
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported(),flush=True)
B,L,N=128,200,3706
tokens=torch.randint(1,N+1,(B,L+1),device=device)

def sync_time(label, fn):
    torch.cuda.synchronize(); t=time.perf_counter(); out=fn(); torch.cuda.synchronize(); dt=time.perf_counter()-t
    print(label,round(dt,3),'s',flush=True); return out,dt

def profile(cls,name):
    print('\n===',name,'===',flush=True)
    m=cls(N,200,d=64,layers=2,side=256,h=16,active=8,top_side=2,degree=4).cuda().train()
    # warmup short path
    with torch.autocast('cuda',dtype=torch.bfloat16): m.encode(tokens[:4,:20])
    torch.cuda.synchronize()
    m.zero_grad(set_to_none=True)
    def enc():
        with torch.autocast('cuda',dtype=torch.bfloat16): return m.encode(tokens[:,:-1])
    H,tf=sync_time(name+' encode forward',enc)
    del H; torch.cuda.empty_cache()
    m.zero_grad(set_to_none=True)
    def mk_loss():
        with torch.autocast('cuda',dtype=torch.bfloat16): return ar_training_loss(m,tokens,loss_mode='full')
    loss,tl=sync_time(name+' full loss forward',mk_loss)
    _,tb=sync_time(name+' backward',lambda: loss.backward())
    print(name,'loss',float(loss),'TOTAL train-batch',round(tl+tb,3),'s',flush=True)
    del loss,m; torch.cuda.empty_cache()
    return {'encode':tf,'loss_forward':tl,'backward':tb,'train_batch':tl+tb}

base=profile(SparseWalker,'BASE')
temp=profile(SparseWalkerTemporalMemory,'TEMPORAL')
print('\nRATIO temporal/base', {k:round(temp[k]/base[k],2) for k in base}, flush=True)
